# 07 - Baseline Predictive Model and Leakage Check

**Purpose:** Fit a standard XGBoost response model predicting visit probability ($P(Y=1 \mid X, T)$), inspect feature importances, and verify the absence of train/test data leakage.

**Inputs:** `../data/train.parquet`, `../data/test.parquet`.

**Outputs:** Baseline model evaluation metrics (ROC-AUC 0.6152) and feature importances.

In [1]:
# ==========================================
# Load Partitions & Feature Selection
# ==========================================
import pandas as pd

train_df = pd.read_parquet('../data/train.parquet')
test_df = pd.read_parquet('../data/test.parquet')

feature_cols = [c for c in train_df.columns if c not in ['visit', 'treatment_flag', 'spend']]

X_train, y_train, t_train = train_df[feature_cols], train_df['visit'], train_df['treatment_flag']
X_test, y_test, t_test = test_df[feature_cols], test_df['visit'], test_df['treatment_flag']

print(X_train.shape, X_test.shape)
print(feature_cols)

(29885, 11) (12808, 11)
['recency', 'history', 'mens', 'womens', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web', 'zip_code_Rural', 'zip_code_Surburban', 'zip_code_Urban']


In [2]:
# ==========================================
# Fit Baseline Response Model
# ==========================================
from xgboost import XGBClassifier

X_train_with_t = X_train.assign(treatment=t_train)
X_test_with_t = X_test.assign(treatment=t_test)

baseline = XGBClassifier(random_state=42, eval_metric='logloss').fit(X_train_with_t, y_train)

In [3]:
# ==========================================
# Model Evaluation & Feature Importance
# ==========================================
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

y_pred_proba = baseline.predict_proba(X_test_with_t)[:, 1]
y_pred = baseline.predict(X_test_with_t)

print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1:        {f1_score(y_test, y_pred):.4f}")

importances = pd.Series(baseline.feature_importances_, index=X_train_with_t.columns).sort_values(ascending=False)
print("\nFeature importances:")
print(importances)

ROC-AUC:   0.6152
Precision: 0.2410
Recall:    0.0122
F1:        0.0232

Feature importances:
womens                  0.152712
newbie                  0.147527
treatment               0.100344
zip_code_Rural          0.088610
mens                    0.077780
channel_Web             0.077046
recency                 0.067513
history                 0.062847
channel_Phone           0.061813
channel_Multichannel    0.058513
zip_code_Surburban      0.052680
zip_code_Urban          0.052616
dtype: float32


This baseline model predicts overall visit propensity, which is distinct from incremental uplift. An ROC-AUC of 0.6152 confirms the covariates provide meaningful signal above random guessing (0.50) without unrealistically high performance that would signal data leakage. At the default 0.5 classification threshold, precision (0.24) and recall (0.01) are heavily skewed by the unweighted ~13% base visit rate.

In the feature importance ranking, `womens` and `newbie` rank above `treatment` (0.100). The model detects the campaign's effect, but the model relies more on womens and newbie than on treatment when predicting visits. Crucially, a response model estimates who visits, but cannot determine who visits specifically because of the email versus who would visit anyway.

**Leakage check:** The one-hot encoding in `06_feature_engineering.ipynb` was applied to the full dataset before splitting. One-hot encoding learns nothing from the data beyond the category labels (channel and zip_code have the same values in train and test), so doing it before the split cannot leak information. Transformations that learn statistics, such as scaling or target encoding, would have to be fit on the training rows only. Furthermore, `X_test` remained completely untouched during model training; `X_test_with_t` was assembled solely for evaluation after fitting `baseline` on `X_train_with_t`.